# Rastreo de meses — Universo de Sherly (PD BHV PYME)

**Objetivo:** determinar, para cada tabla fuente, de qué mes toma la data la tabla original construida por Sherly (el "target").
**Método:** para un cliente cuyo valor *cambia* entre el mes de proceso y el mes anterior, comparamos el valor del target contra ambos meses de la tabla fuente. El mes cuyo valor coincide es el mes del que Sherly tomó la data.
Se necesita un cliente con valores DISTINTOS entre meses; si el valor no cambia, no se puede distinguir el mes (resultado ambiguo).

In [0]:
from pyspark.sql import functions as F

In [0]:
# ===============================================================
# PARÁMETROS  (ajustar según el período a validar)
# ===============================================================
TARGET_TABLE   = "catalog_lhcl_prod_bcp_expl.bcp_edv_mmgr.t06086_pd_bhv_cliente_pyme_202306_202604_32var"
SRC_CATALOG    = "catalog_lhcl_prod_bcp"
CODMES_PROCESO = 202604     # mes del proceso (el que etiqueta la tabla target)
CODMES_ANTERIOR = 202603    # mes de proceso - 1

df_target = spark.table(TARGET_TABLE)

In [0]:
# ===============================================================
# FUNCIÓN DE RASTREO
# ===============================================================
def rastrear_mes(nombre, tabla_fuente, campo_fuente, campo_target, key):
    """
    Determina de qué mes (proceso vs anterior) proviene la data de una variable
    en la tabla target de Sherly.

    Parámetros:
      nombre        : etiqueta para el reporte
      tabla_fuente  : tabla origen completa (catalog.schema.tabla)
      campo_fuente  : nombre del campo en la tabla origen
      campo_target  : nombre del campo (con prefijo) en la tabla target de Sherly
      key           : clave de cruce (codclaveunicocli o codinternocomputacional)
    """
    print(f"\n=== {nombre} (key={key}) ===")
    f = spark.table(tabla_fuente)

    # 1. Tomar la fuente en cada mes y quedarnos con el valor por cliente
    f_ant = (f.filter(F.col("codmes") == CODMES_ANTERIOR)
               .select(key, F.col(campo_fuente).alias("v_ant")))
    f_pro = (f.filter(F.col("codmes") == CODMES_PROCESO)
               .select(key, F.col(campo_fuente).alias("v_pro")))

    # 2. Quedarnos SOLO con clientes cuyo valor cambia entre los dos meses
    #    (si no cambia, no permite distinguir el mes -> ambiguo)
    difer = (f_ant.join(f_pro, key, "inner")
                  .filter(F.col("v_ant").isNotNull() & F.col("v_pro").isNotNull())
                  .filter(F.col("v_ant") != F.col("v_pro"))
                  .limit(20))

    # 3. Cruzar esos clientes con el target y ver con qué mes coincide el valor
    tgt = (df_target.filter(F.col("codmes") == CODMES_PROCESO)
                    .select(key, F.col(campo_target).alias("v_target")))
    res = difer.join(tgt, key, "inner").limit(5).collect()

    if not res:
        print("  (sin clientes con diferencia + presencia en target -> AMBIGUO)")
        return

    # 4. Para cada caso, decidir el mes ganador
    for row in res:
        if row["v_target"] == row["v_ant"]:
            veredicto = "MES ANTERIOR"
        elif row["v_target"] == row["v_pro"]:
            veredicto = "MES PROCESO"
        else:
            veredicto = "NINGUNO"
        print(f"  target={row['v_target']} | anterior={row['v_ant']} | proceso={row['v_pro']} -> {veredicto}")

In [0]:
# ===============================================================
# RASTREO POR TABLA
# Cada bloque mapea: variable target  <->  campo y tabla fuente
# ===============================================================

# --- Familia MATRICES (schema matrizvariables_vu) ---
# Hipótesis: leen del MES DE PROCESO
rastrear_mes(
    "MTX_MOV_CARGO_PAS",
    f"{SRC_CATALOG}.bcp_ddv_matrizvariables_vu.hm_matrizmovimientocargopasivo",
    "isav_tkt_opec_pago_srv_prm_u3m",
    "MTX_MOV_CARGO_PAS__isav_tkt_opec_pago_srv_prm_u3m",
    "codclaveunicocli",
)

rastrear_mes(
    "MTX_RESUMEN_ACT_PAS",
    f"{SRC_CATALOG}.bcp_ddv_matrizvariables_vu.hm_matrizresumensaldoactivopasivo",
    "prod_mto_sld_fim_pas_min_24_24_rt_u24",
    "MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_pas_min_24_24_rt_u24",
    "codclaveunicocli",
)

# --- Familia APP PYME / módulos (schema rbmbcapym_apppyme_vu) ---
# Hipótesis: leen del MES ANTERIOR
rastrear_mes(
    "MOD_DEMO",
    f"{SRC_CATALOG}.bcp_ddv_rbmbcapym_apppyme_vu.hm_scoreappbasepymemodulodemografico",
    "ctdmesantiguedadempsunat",
    "MOD_DEMO__ctdmesantiguedadempsunat",
    "codclaveunicocli",
)

rastrear_mes(
    "MOD_ACT",
    f"{SRC_CATALOG}.bcp_ddv_rbmbcapym_apppyme_vu.hm_scoreappbasepymemoduloactivo",
    "isav_mto_opea_estvta_pym_u6m_rt_max_u12",
    "MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12",
    "codclaveunicocli",
)

# --- Familia VIDEA variables modelos (schema adrmmgr_videavariablesmodelos_vu) ---
# Resultado: NO uniforme dentro del mismo schema -> rastrear cada una
rastrear_mes(
    "VIDEVAR_MORA_POND",
    f"{SRC_CATALOG}.bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_matrizmoraponderadaclientemmgr",
    "mtodeudaclasifriesgofactordsctosolu12",
    "VIDEVAR_MTX_MORA_POND_CLI_MMGR__mtodeudaclasifriesgofactordsctosolu12",
    "codclaveunicocli",
)

rastrear_mes(
    "CLASI_EXPER_CLI",
    f"{SRC_CATALOG}.bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_clasificacionclientenivelexperienciapyme",
    "ctdempleado",
    "CLASI_EXPER_CLI__ctdempleado",
    "codclaveunicocli",
)

rastrear_mes(
    "EVOL_CLI_PYM",
    f"{SRC_CATALOG}.bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_variableactivoevolucionclientepyme",
    "ctdmaxdiamorau6m",
    "EVOL_CLI_PYM__ctdmaxdiamorau6m",
    "codinternocomputacional",   # esta tabla cruza por codinternocomputacional
)

rastrear_mes(
    "PASIVO_EVOL_SALD",
    f"{SRC_CATALOG}.bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_variablepasivoevolucionsaldopyme",
    "mtoprmincrvariacionmensualprmvigsolu6m",
    "PASIVO_EVOL_SALD_PYM__mtoprmincrvariacionmensualprmvigsolu6m",
    "codinternocomputacional",
)

## Resultado del rastreo (período 202604)
| Tabla | Schema | Mes del que toma la data |
|-------|--------|--------------------------|
| Recableo (APP_SCORE_APROB_PYME__*) | rbmbcapym_apppyme_vu | **Mes anterior** |
| MOD_DEMO | rbmbcapym_apppyme_vu | **Mes anterior** |
| MOD_ACT | rbmbcapym_apppyme_vu | **Mes anterior** |
| VIDEVAR_MORA_POND | adrmmgr_videavariablesmodelos_vu | **Mes anterior** |
| PASIVO_EVOL_SALD | adrmmgr_videavariablesmodelos_vu | **Mes anterior** |
| Todas las MTX_* | matrizvariables_vu | **Mes de proceso** |
| EVOL_CLI_PYM | adrmmgr_videavariablesmodelos_vu | **Mes de proceso** |
| CLASI_EXPER_CLI | adrmmgr_videavariablesmodelos_vu | **Mes de proceso** |

**Conclusión:** el desfase NO es uniforme. El universo de clientes sale del
portafolio del mes de proceso, mientras que las variables provienen de meses
distintos según la tabla. Esto es lo que la clase UniversoImplementacion
reproduce mediante el parámetro `mes_lectura` en cada método de lectura.